# Agentic Evals: from agent trace to release decision

`agentic-evals` is a framework-agnostic evaluation engine for LLM and agent outputs. It checks both **what the agent said** and **how it got there**.

This demo answers three questions:

1. What is this repository for?
2. Why are ordinary output tests not enough for agents?
3. How can evaluation results stop a bad agent version from being released?

The example uses a support agent that looks up an order. It needs to return a useful answer, call the right tool with the right argument, and stay within latency and cost limits.

## Why use an evaluation engine?

A normal unit test might assert that an answer contains `shipped`. An agent can still be unsafe or expensive while passing that assertion: it may call an unrelated tool, omit a required argument, repeat a tool call, take too long, or cost too much.

This repository gives those checks one consistent model:

- `TestSuite` and `TestCase` describe expectations.
- `EvaluationSample` contains the agent output and a small trace shape.
- Built-in scorers evaluate text and tool trajectories.
- `EvaluationReport` summarizes case-level results, latency, and cost.
- `evaluate_gate` converts the report into a release decision.

It is deliberately independent of a tracing vendor or agent framework, so existing application traces can be adapted into `EvalTrace` and `EvalSpan`.

In [ ]:
from pathlib import Path
import sys

# Make the checkout importable when this notebook is opened from the repo.
repo_root = Path.cwd()
if (repo_root / "src").is_dir():
    sys.path.insert(0, str(repo_root / "src"))

from agentic_evals import (
    EvalSpan,
    EvalTrace,
    EvaluationSample,
    EvaluatorConfig,
    GateConfig,
    TestCase,
    TestSuite,
    builtin_scorer_names,
    default_registry,
    evaluate_gate,
    evaluate_suite,
)

## 1. Define what good looks like

The suite below combines deterministic requirements with trajectory scorers. The evaluator registry resolves named built-in scorers without requiring an LLM call.

In [ ]:
suite = TestSuite(
    name="order-support-agent",
    version="2026.09",
    description="Quality and operational checks for an order-status agent.",
    cases=[
        TestCase(
            id="order-status-001",
            name="Returns the order status safely",
            input="Where is order 1042?",
            expected_contains=["shipped", "2026-09-15"],
            required_output_fields=["answer", "order_id", "status"],
            required_tools=["order_lookup"],
            required_tool_arguments={"order_lookup": ["order_id"]},
            max_latency_ms=1000,
            max_cost_usd=0.02,
            max_turns=3,
            evaluators=[
                EvaluatorConfig(
                    name="tool_call_precision",
                    threshold=1.0,
                    config={"allowed_tools": ["order_lookup", "respond"]},
                ),
                EvaluatorConfig(name="tool_call_recall", threshold=1.0),
                EvaluatorConfig(
                    name="trajectory_efficiency",
                    threshold=0.8,
                    config={"baseline_latency_ms": 1500, "baseline_cost_usd": 0.03},
                ),
            ],
        )
    ],
)

registry = default_registry()
print(suite.name, "contains", len(suite.cases), "case")
print("Available built-in scorers:", len(registry._evaluators))

## 2. Represent an agent run

The package does not require a particular observability product. We provide only the fields the evaluators need: output text, tool spans, latency, cost, and metadata. The rest of an application's trace can remain in its own system.

In [ ]:
healthy_sample = EvaluationSample(
    case_id="order-status-001",
    output="{\"answer\": \"Order 1042 shipped and is expected on 2026-09-15.\", \"order_id\": 1042, \"status\": \"shipped\"}",
    trace=EvalTrace(
        trace_id="trace-good-001",
        total_latency_ms=420,
        estimated_cost_usd=0.012,
        metadata={"turn_count": 2},
        spans=[
            EvalSpan(tool_name="order_lookup", attributes={"tool_args": {"order_id": 1042}}),
            EvalSpan(tool_name="respond"),
        ],
    ),
)

report = evaluate_suite(suite, [healthy_sample], registry=registry)
report.summary.model_dump()

In [ ]:
def show_report(report):
    summary = report.summary
    print(f"Pass rate: {summary.pass_rate:.0%}")
    print(f"Average score: {summary.average_score:.3f}")
    print(f"Average latency: {summary.average_latency_ms:.0f} ms")
    print(f"Total cost: ${summary.total_cost_usd:.3f}")
    for case in report.cases:
        print(f"\n{case.case_name}: {'PASS' if case.passed else 'FAIL'}")
        for score in case.scores:
            status = "PASS" if score.passed else "FAIL"
            print(f"  {status:4} {score.name}: {score.value:.3f} - {score.explanation}" )

show_report(report)

The healthy run passes because the answer contains the expected facts, the JSON fields are present, `order_lookup` was called with `order_id`, and the run stays under the configured operational limits. Notice that the report includes both quality scores and production signals.

## 3. Turn the report into a release gate

A report is useful for diagnosis. A gate is useful for automation: CI can fail a deployment when quality drops, latency regresses, or cost becomes unavailable or too high.

In [ ]:
gate_config = GateConfig(
    min_pass_rate=1.0,
    min_average_score=0.95,
    max_failed_cases=0,
    max_average_latency_ms=1000,
    max_total_cost_usd=0.02,
)

decision = evaluate_gate(report, gate_config)
print("Release gate:", "PASS" if decision.passed else "FAIL")
print("Observed:", decision.observed)
print("Reasons:", decision.reasons or ["All thresholds met."])

## 4. See the failure a text-only test would miss

This regression returns the same general status, but it uses the wrong tool, omits the required argument, takes too long, costs too much, and exceeds the turn budget. The gate catches the operational and trajectory problems before release.

In [ ]:
regression_sample = EvaluationSample(
    case_id="order-status-001",
    output="{\"answer\": \"Order 1042 shipped.\", \"order_id\": 1042, \"status\": \"shipped\"}",
    trace=EvalTrace(
        trace_id="trace-regression-001",
        total_latency_ms=1800,
        estimated_cost_usd=0.031,
        metadata={"turn_count": 5},
        spans=[
            EvalSpan(tool_name="web_search"),
            EvalSpan(tool_name="web_search"),
            EvalSpan(tool_name="respond"),
        ],
    ),
)

regression_report = evaluate_suite(
    suite, [regression_sample], registry=registry
)
show_report(regression_report)

regression_decision = evaluate_gate(regression_report, gate_config)
print("\nRelease gate:", "PASS" if regression_decision.passed else "FAIL")
for reason in regression_decision.reasons:
    print(" -", reason)

## Takeaway

Use `agentic-evals` when an agent needs more than a string assertion. It provides a small, portable contract for evaluating answer quality, structured output, tool use, latency, cost, and turn count, then makes those results enforceable in CI or deployment workflows.

For production use, replace the hand-built `EvaluationSample` with samples collected from your application, or use `run_live_suite` with a trusted Python or HTTP target. For model-based judging, register an `LLMJudgeEvaluator` or `LLMRubricEvaluator` with your own provider call; the package remains provider-neutral.